# Numerical Feature Transformations — Solution Notebook

Companion to the Practice Skeleton. All TODOs are completed, alternate implementations are shown, and a full simulation section is included.

**Data:** `data/starbucks_customers.csv`

**Cheat Sheet** (same as skeleton) is embedded for convenience; see also `Numerical_Feature_Transformations_Cheatsheet.docx`.

**Audience guidance**
- **Data / ML practitioners** – study the sklearn vs pure-NumPy alternatives, the Pipeline stretch, and the simulation code.
- **Business analysts** – focus on the printed means/stds and the before/after histograms.
- **Executives / non-specialists** – the comparison chart and the final one-sentence takeaway are the main deliverables.


## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
# Display settings
pd.set_option('display.precision', 4)
%matplotlib inline


## 1. Load & basic exploration

In [ ]:
coffee = pd.read_csv('data/starbucks_customers.csv')
print(coffee.columns.tolist())
print(coffee.info())


In [ ]:
print(coffee.describe())
print('\nSkewness:')
print(coffee.skew())
# Age is clearly right-skewed (~1.76); spent and nearest_starbucks mildly so.


## 2. Centering

In [ ]:
ages = coffee['age']
min_age = np.min(ages)
max_age = np.max(ages)
print(f'min_age = {min_age}, max_age = {max_age}, range = {max_age - min_age}')

mean_age = np.mean(ages)
print(f'mean_age = {mean_age:.4f}')

centered_ages = ages - mean_age
print(centered_ages.head(10))

plt.figure(figsize=(6,4))
plt.hist(centered_ages, bins=12, color='#66BB6A', edgecolor='white')
plt.axvline(0, color='red', linestyle='--', label='new mean = 0')
plt.title('Centered Age')
plt.xlabel('Distance from mean age')
plt.ylabel('Count')
plt.legend()
plt.show()
print('Centered mean (should be ~0):', centered_ages.mean())


In [ ]:
# Optional: center nearest_starbucks
dist = coffee['nearest_starbucks']
centered_dist = dist - dist.mean()
fig, ax = plt.subplots(1,2, figsize=(10,3.5))
ax[0].hist(centered_ages, bins=10, color='#66BB6A', edgecolor='white')
ax[0].set_title('Centered Age')
ax[1].hist(centered_dist, bins=8, color='#42A5F5', edgecolor='white')
ax[1].set_title('Centered nearest_starbucks')
for a in ax: a.axvline(0, color='red', ls='--')
plt.tight_layout(); plt.show()


## 3. Standardization

In [ ]:
# 3.1 Manual
mean_age = np.mean(ages)
std_dev_age = np.std(ages)          # population std (ddof=0) to match sklearn default
ages_standardized = (ages - mean_age) / std_dev_age
print('Manual standardization – mean:', np.mean(ages_standardized))
print('Manual standardization – std :', np.std(ages_standardized))


In [ ]:
# 3.2 sklearn
scaler = StandardScaler()
ages_reshaped = np.array(ages).reshape(-1, 1)
ages_scaled = scaler.fit_transform(ages_reshaped)
print('sklearn StandardScaler – mean:', np.mean(ages_scaled))
print('sklearn StandardScaler – std :', np.std(ages_scaled))
print('scaler.mean_, scaler.scale_   :', scaler.mean_, scaler.scale_)


## 4. Min-Max Normalization

In [ ]:
spent = coffee['spent']
min_spent = np.min(spent)
max_spent = np.max(spent)
spent_range = max_spent - min_spent
print(f'min={min_spent}, max={max_spent}, range={spent_range}')

spent_normalized = (spent - min_spent) / spent_range
print(spent_normalized.describe())
print('Unique values (sorted):', np.sort(spent_normalized.unique()))


In [ ]:
mm_scaler = MinMaxScaler()
spent_mm = mm_scaler.fit_transform(np.array(spent).reshape(-1,1)).ravel()
print('MinMaxScaler result min/max:', spent_mm.min(), spent_mm.max())


## 5. Log transformation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
axes[0,0].hist(coffee['spent'], bins=12, color='#42A5F5', edgecolor='white')
axes[0,0].set_title(f"spent original (skew={coffee['spent'].skew():.2f})")
axes[0,1].hist(coffee['age'], bins=12, color='#42A5F5', edgecolor='white')
axes[0,1].set_title(f"age original (skew={coffee['age'].skew():.2f})")

log_spent = np.log1p(coffee['spent'])
log_age   = np.log1p(coffee['age'])
axes[1,0].hist(log_spent, bins=12, color='#26A69A', edgecolor='white')
axes[1,0].set_title(f"log1p(spent) (skew={log_spent.skew():.2f})")
axes[1,1].hist(log_age, bins=12, color='#26A69A', edgecolor='white')
axes[1,1].set_title(f"log1p(age) (skew={log_age.skew():.2f})")
plt.tight_layout(); plt.show()


## 6. Binning

In [ ]:
coffee['age_bin'] = pd.cut(coffee['age'], bins=3,
                           labels=['young', 'mid', 'senior'])
print(coffee['age_bin'].value_counts().sort_index())


In [ ]:
coffee['spent_q'] = pd.qcut(coffee['spent'], q=4,
                            labels=['Q1','Q2','Q3','Q4'])
print(coffee['spent_q'].value_counts().sort_index())
# show the actual bin edges
print(pd.qcut(coffee['spent'], q=4).cat.categories)


## 7. Alternate implementations

### Standardization – pure pandas


In [ ]:
# Alternate 1: pandas vectorized (identical math)
z_pandas = (coffee['age'] - coffee['age'].mean()) / coffee['age'].std(ddof=0)
print('pandas z mean/std:', z_pandas.mean(), z_pandas.std(ddof=0))

# Alternate 2: using scipy.stats.zscore if available
try:
    from scipy.stats import zscore
    z_scipy = zscore(coffee['age'], ddof=0)
    print('scipy z mean/std:', z_scipy.mean(), z_scipy.std())
except ImportError:
    print('scipy not installed – skipped')


### Min-Max – alternative range

In [ ]:
# Map spent into [-1, 1] instead of [0, 1]
mm_m1_1 = MinMaxScaler(feature_range=(-1, 1))
spent_m1_1 = mm_m1_1.fit_transform(coffee[['spent']]).ravel()
print('range [-1,1] min/max:', spent_m1_1.min(), spent_m1_1.max())


### Binning with np.digitize

In [ ]:
# Equal-width bins via numpy
edges = np.linspace(coffee['age'].min(), coffee['age'].max(), 4)
age_dig = np.digitize(coffee['age'], edges[1:-1])   # 0,1,2
print('np.digitize counts:', np.bincount(age_dig))
print('edges used:', edges)


## 8. More Practice – multi-feature scaling & simple model

In [ ]:
features = ['age', 'nearest_starbucks', 'spent']
X_raw = coffee[features].copy()

scaler_multi = StandardScaler()
X_std = pd.DataFrame(scaler_multi.fit_transform(X_raw),
                     columns=[f + '_std' for f in features])
print(X_std.describe().loc[['mean','std']])

# Simple model: predict spent from age + nearest (raw vs scaled)
y = coffee['spent'].values
X1 = coffee[['age', 'nearest_starbucks']].values
X2 = StandardScaler().fit_transform(X1)

r2_raw = LinearRegression().fit(X1, y).score(X1, y)
r2_std = LinearRegression().fit(X2, y).score(X2, y)
print(f'R² raw features: {r2_raw:.4f}')
print(f'R² standardized: {r2_std:.4f}')
# Note: for ordinary least squares the R² is invariant to linear rescaling,
# so the values are essentially identical. Scaling matters more for regularized
# models, distance-based algorithms, and neural nets.


In [ ]:
# Pipeline stretch
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])
pipe.fit(X1, y)
print('Pipeline R²:', pipe.score(X1, y))


## 9. Simulation – sensitivity of model fit to transform choice

We repeatedly bootstrap the data, optionally inject Gaussian noise, apply each transform, and record in-sample R² of a linear model predicting `spent`.


In [ ]:
np.random.seed(42)

# Build design matrix
X_df = coffee[['age', 'nearest_starbucks']].copy()
X_df['avg_rating'] = coffee[['rate_quality','rate_price','rate_promo',
                             'ambiance','wifi','service']].mean(axis=1)
y = coffee['spent'].values

def simulate(transform_name, noise_std=0.3, n_sims=80, sample_frac=1.0):
    scores = []
    n = int(len(y) * sample_frac)
    for _ in range(n_sims):
        idx = np.random.choice(len(y), size=n, replace=True)
        Xb = X_df.iloc[idx].values.astype(float)
        yb = y[idx]
        Xb = Xb + np.random.normal(0, noise_std, Xb.shape)
        if transform_name == 'raw':
            Xt = Xb
        elif transform_name == 'standard':
            Xt = StandardScaler().fit_transform(Xb)
        elif transform_name == 'minmax':
            Xt = MinMaxScaler().fit_transform(Xb)
        elif transform_name == 'log_age':
            Xt = Xb.copy()
            Xt[:, 0] = np.log1p(np.clip(Xt[:, 0], 0, None))
            Xt = StandardScaler().fit_transform(Xt)
        else:
            raise ValueError(transform_name)
        try:
            scores.append(LinearRegression().fit(Xt, yb).score(Xt, yb))
        except Exception:
            pass
    return scores

transforms = ['raw', 'standard', 'minmax', 'log_age']
results = {t: simulate(t) for t in transforms}

fig, ax = plt.subplots(figsize=(9, 5))
bp = ax.boxplot([results[t] for t in transforms],
                tick_labels=['Raw', 'StandardScaler', 'MinMaxScaler', 'Log(age)+Std'],
                patch_artist=True)
colors = ['#90CAF9', '#A5D6A7', '#FFCC80', '#CE93D8']
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c)
ax.set_ylabel('In-sample R² (80 bootstrap + noise runs)')
ax.set_title('Simulation: Effect of Numerical Transforms on Linear Model Fit')
plt.tight_layout()
plt.show()

for t in transforms:
    print(f'{t:12s}  mean R² = {np.mean(results[t]):.4f}  std = {np.std(results[t]):.4f}')


### Experiment: change noise_std and sample size
Try `noise_std = 0`, `1.5`, `3.0` and `sample_frac = 0.4`. Observe when the log transform helps most (usually when age skew interacts with noise).


In [ ]:
# Example re-run with higher noise and smaller sample
results_high_noise = {t: simulate(t, noise_std=2.0, sample_frac=0.5) for t in transforms}
print('Higher noise + half sample:')
for t in transforms:
    print(f'{t:12s}  mean R² = {np.mean(results_high_noise[t]):.4f}')


## 10. Flowchart

See `numerical_transformations_flowchart.png` (also displayed below if the file is present).


In [ ]:
from IPython.display import Image, display
import os
path = 'numerical_transformations_flowchart.png'
if os.path.exists(path):
    display(Image(path, width=700))
else:
    print('Flowchart image not found in working directory – open it from the project folder.')


## 11. Reflection & audience adaptation (model answers)

- **First transform for a new similar data set:** Start with a quick skew/scale diagnosis, then apply `log1p` to any strongly right-skewed positive features and `StandardScaler` to the whole numeric matrix. This combination is the safest default for linear models, KNN and regularized regression.
- **Store-manager explanation:** “Right now the model treats ‘age = 50’ as five times more important than ‘distance = 10’ simply because the numbers are larger. Standardization puts every variable on the same ruler (average = 0, typical spread = 1) so the model can fairly decide which factors really matter for spend.”
- **Risk of min-max:** Because the min and max are estimated from the observed data, a single extreme outlier (or a new customer whose spend exceeds the training max) can compress the rest of the feature into a tiny interval or produce values outside [0,1] at prediction time. The simulation with added noise sometimes shows slightly higher variance for MinMax than for StandardScaler.
